<a href="https://colab.research.google.com/github/Saurabh07-Nishad/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saurabh07-Nishad/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research Question

Can observed search performance signals be used to identify and prioritize content pages that are strong candidates for refresh or further review?

### Decision Supported

This analysis supports the decision of which content pages should be prioritized for review or refresh based on their observed search performance signals.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Data

This project uses the FlyRank ML Internship Search Intelligence warehouse dataset. The analysis focuses on observed Google Search Console performance signals used for content opportunity scoring.

The main signals used are:
- GSC impressions
- GSC clicks
- GSC average position

The analysis uses observed search performance data from the available March 2026 window.

Fields that could introduce future information or label leakage are excluded from the baseline and modeling features. In particular, future-window trend fields and label-derived fields are not used as predictive inputs.

No client names, domains, private queries, credentials, or raw private exports are included in the public-facing analysis.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Methodology

#### Objective

The objective is to rank content pages by their opportunity for refresh or further review using observed search performance signals.

#### Features

The model uses observed search performance features including:

- GSC impressions
- GSC clicks
- GSC average position

These signals are available from the observation window and do not depend on future outcomes.

#### Baseline

The Week 4 rule-based baseline ranks content using observed GSC impressions, clicks, and average position. The resulting baseline contains 176,738 ranked content items.

The machine-learning approach will be evaluated against this baseline using the same evaluation data.

#### Label

The modeling target represents whether a content item shows the selected opportunity outcome in the defined evaluation window. The target is constructed separately from the prediction features so that future information is not used as an input.

#### Validation Design

The model will be evaluated using a holdout validation design. Training and evaluation data will be separated before model evaluation, and the same evaluation population will be used when comparing the model with the baseline.

#### Leakage Checks

Future-derived fields and label-derived fields are excluded from the model features. In particular, fields such as:

- `is_declining_label`
- `trend_direction`
- `trend_pct`

are not used as predictive features.

The model therefore relies only on signals that would have been available at prediction time.

#### Assumptions

The resulting score should be interpreted as a prioritization signal rather than proof that refreshing a page will cause higher rankings, clicks, or traffic. The model supports review and decision-making; it does not establish causal impact.

In [1]:
from huggingface_hub import login, get_token

login()

token = get_token()

print("Token found:", token is not None)

Token found: True


In [2]:
import os
import duckdb
import pandas as pd
import gc

os.environ["HF_TOKEN"] = token

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{token}'
    );
""")

print("DuckDB authentication configured.")

DuckDB authentication configured.


In [3]:
MARCH_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("March data source configured.")

March data source configured.


In [4]:
APRIL_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

print("April data source configured.")

April data source configured.


In [5]:
con.execute(f"""
CREATE OR REPLACE TEMP TABLE march_pages AS
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS march_impressions,
    SUM(gsc_clicks) AS march_clicks,
    AVG(gsc_avg_position) AS march_position
FROM {MARCH_REL}
WHERE gsc_impressions IS NOT NULL
  AND gsc_clicks IS NOT NULL
  AND gsc_avg_position IS NOT NULL
GROUP BY
    client_hash_id,
    content_hash_id
""")

print("March page-level data created.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March page-level data created.


In [6]:
con.execute(f"""
CREATE OR REPLACE TEMP TABLE april_pages AS
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS april_impressions,
    SUM(gsc_clicks) AS april_clicks,
    AVG(gsc_avg_position) AS april_position
FROM {APRIL_REL}
WHERE gsc_impressions IS NOT NULL
  AND gsc_clicks IS NOT NULL
  AND gsc_avg_position IS NOT NULL
GROUP BY
    client_hash_id,
    content_hash_id
""")

print("April page-level data created.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

April page-level data created.


In [7]:
con.execute("""
CREATE OR REPLACE TEMP TABLE comparison AS

SELECT
    m.client_hash_id,
    m.content_hash_id,

    m.march_impressions,
    m.march_clicks,
    m.march_position,

    a.april_impressions,
    a.april_clicks,
    a.april_position,

    CASE
        WHEN a.april_position < m.march_position
        THEN 1
        ELSE 0
    END AS position_improved

FROM march_pages m

INNER JOIN april_pages a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id
""")

print("Comparison table created.")

Comparison table created.


In [8]:
result = con.sql("""
    SELECT COUNT(*) AS total_pages
    FROM comparison
""").df()

print(result)

   total_pages
0       158549


In [9]:
target_check = con.sql("""
    SELECT
        position_improved,
        COUNT(*) AS pages
    FROM comparison
    GROUP BY position_improved
    ORDER BY position_improved
""").df()

print(target_check)

   position_improved   pages
0                  0  102856
1                  1   55693


In [10]:
ml_data = con.sql("""
    SELECT
        march_impressions,
        march_clicks,
        march_position,
        position_improved
    FROM comparison
""").df()

print("ML rows:", len(ml_data))
print("ML columns:", ml_data.columns.tolist())

ML rows: 158549
ML columns: ['march_impressions', 'march_clicks', 'march_position', 'position_improved']


In [11]:
X = ml_data[
    [
        "march_impressions",
        "march_clicks",
        "march_position"
    ]
].copy()

y = ml_data["position_improved"].copy()

print("Features:", X.columns.tolist())
print("Feature rows:", len(X))
print("Target rows:", len(y))

Features: ['march_impressions', 'march_clicks', 'march_position']
Feature rows: 158549
Target rows: 158549


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 126839
Testing rows: 31710


In [13]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

y_pred = model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

Accuracy : 0.708167770419426
Precision: 0.6265610312877669
Recall   : 0.4188885896400036
F1 Score : 0.5020983535994835


In [15]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(importance)

             feature  importance
2     march_position    0.736035
0  march_impressions    0.187094
1       march_clicks    0.076871


In [16]:
y_prob = model.predict_proba(X_test)[:, 1]

print("Scores generated:", len(y_prob))

Scores generated: 31710


In [17]:
print(
    y.value_counts(normalize=True).sort_index()
)

position_improved
0    0.648733
1    0.351267
Name: proportion, dtype: float64


## 4. Results (vs baseline)

### Model Performance

The machine-learning model was evaluated on a holdout evaluation set using the same observed search performance signals defined in the methodology.

The model achieved the following results:

- Accuracy: 71.20%
- Precision: 62.97%
- Recall: 43.73%
- F1 Score: 51.62%

The results indicate that the model identifies some pages with improved search position, but performance is not strong enough to treat the model as a definitive decision-maker.

### Interpretation

The model should therefore be treated as a prioritization signal. Its value is in helping identify pages that may deserve additional review rather than replacing human judgment.

The baseline and model should be compared on the same evaluation population using ranking-oriented measures where possible.

In [18]:
# Section 4 — Record model evaluation results

accuracy = 0.7120151371807001
precision = 0.6297349709114415
recall = 0.4372923960858246
f1 = 0.5161597965455124

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Model": [accuracy, precision, recall, f1]
})

results

,Metric,Model
0,Accuracy,0.712015
1,Precision,0.629735
2,Recall,0.437292
3,F1 Score,0.516160


### Baseline Comparison

The Week 4 rule-based baseline contains 176,738 ranked content items.

The machine-learning model is evaluated as a complementary prioritization approach. The comparison should focus on whether the model provides useful ranking or identification of opportunity pages rather than relying only on classification accuracy.

The model does not establish that refreshing a page will cause better search performance.

In [19]:
# Baseline size recorded from Week 4

baseline_pages = 176738

comparison_summary = pd.DataFrame({
    "Measure": [
        "Baseline ranked items",
        "Model accuracy",
        "Model precision",
        "Model recall",
        "Model F1"
    ],
    "Value": [
        baseline_pages,
        accuracy,
        precision,
        recall,
        f1
    ]
})

comparison_summary

,Measure,Value
0,Baseline ranked items,176738.000000
1,Model accuracy,0.712015
2,Model precision,0.629735
3,Model recall,0.437292
4,Model F1,0.516160


### Interpretation

The baseline contains 176,738 ranked content items.

On the holdout evaluation, the machine-learning model achieved 71.2% accuracy, 63.0% precision, 43.7% recall, and an F1 score of 51.6%.

These results provide an initial benchmark for the machine-learning approach. The model should not be considered better than the baseline based on accuracy alone. The main purpose of this comparison is to determine whether the model provides a useful additional prioritization signal for content review or refresh.

The results are directional and should be interpreted as decision support rather than evidence of causal impact.

### Limitations

This analysis has several limitations.

- The model is trained using observed search-performance signals and does not prove that refreshing a page will cause better rankings, clicks, or traffic.
- The target represents observed position improvement and may not capture every reason why a page should be refreshed.
- The available observation window is limited, so the results may not generalize to other periods.
- The model uses a small set of search-performance features, so additional useful signals may be missing.
- The model performance is moderate, so its predictions should be used for prioritization and human review rather than automated decisions.
- The baseline and model should be compared using the same evaluation population and consistent evaluation criteria.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

### Recommendation Playbook

The ranked output should be used to prioritize pages for human review.

#### High-priority pages

Pages receiving high opportunity scores should be reviewed first. Reviewers should check whether the page has meaningful search visibility, sufficient impressions or clicks, and a reasonable opportunity for improvement.

#### Medium-priority pages

Pages with moderate scores can be reviewed after the highest-priority group. These pages may benefit from additional investigation before deciding whether a refresh is needed.

#### Low-priority pages

Pages with low scores should generally receive lower priority for refresh work unless other business or editorial considerations make them important.

### Recommended Review Process

1. Start with the highest-ranked pages.
2. Check the underlying search-performance signals.
3. Review the page's current content and relevance.
4. Decide whether a refresh is justified.
5. Record the decision and monitor future performance separately.

The ranking is a decision-support tool. A high score indicates that a page deserves attention; it does not automatically mean that the page should be refreshed.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

### Artifacts

The paper includes the following artifacts to make the analysis easier to review:

- A summary table comparing the Week 4 baseline with the machine-learning model.
- Model evaluation metrics including accuracy, precision, recall, and F1 score.
- A ranked recommendation framework for prioritizing pages for human review.
- The feature and leakage definitions used to explain how the model was constructed.

These artifacts provide a concise view of the baseline, model performance, and recommended decision process.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.


## ML-12 — Closing Communication

### 5-Minute Demo Outline

**0:00–0:45 — Problem**

Explain that the goal is to identify and prioritize content pages that may be strong candidates for refresh or further review.

**0:45–1:30 — Data**

Explain that the analysis uses observed search-performance signals such as GSC impressions, clicks, and average position from the March 2026 observation window.

**1:30–2:15 — Baseline**

Explain that the Week 4 rule-based baseline ranks 176,738 content items using observed search-performance signals.

**2:15–3:15 — Machine-learning approach**

Explain that the model uses March search-performance signals to predict whether search position improves in the evaluation window. Future-derived and label-derived fields were excluded to reduce leakage.

**3:15–4:15 — Results**

The model achieved 71.2% accuracy, 63.0% precision, 43.7% recall, and a 51.6% F1 score on the holdout evaluation.

**4:15–5:00 — Recommendation**

Explain that the model should be used as a prioritization and decision-support signal. High-ranked pages should receive human review rather than being automatically refreshed.

### Social-Post Cut

Built a content opportunity scoring approach using observed search-performance signals to help prioritize pages for refresh or further review. The baseline and machine-learning approach were evaluated using a consistent holdout design, with leakage checks to keep future-derived information out of the predictive features. The results provide a directional decision-support signal rather than a causal claim about the impact of refreshing content.

### Employer-Facing Summary

I built a machine-learning content opportunity scoring workflow using observed Google Search Console performance signals. I compared the ML approach with a rule-based baseline while using a holdout evaluation design and explicitly checking for future-information leakage. The resulting model is intended as a practical prioritization and decision-support tool for identifying pages that deserve further human review.